# bansho — Feature Importance Scoring

This notebook demonstrates `megumi.bansho.score_features` using **synthetic datasets with a known ground truth**. Because we generate the data ourselves, we know exactly which features carry signal — so we can verify that bansho recovers them correctly.

| Example | Generator | Task |
|---------|-----------|------|
| 1 | `make_classification` | Binary classification |
| 2 | `make_regression` | Regression |

**How bansho works:**

1. Two synthetic `RANDOM_1` / `RANDOM_2` columns (standard normal) are injected into the feature matrix.
2. A random forest is fitted on the extended matrix.
3. Mean absolute SHAP values are computed via `TreeExplainer`.
4. Each original feature is labelled relative to the random baselines:

| Label | Condition |
|-------|-----------|
| `predictive` | mean\|SHAP\| > max(RANDOM_1, RANDOM_2) — genuine signal |
| `marginal`   | min < mean\|SHAP\| ≤ max — weak signal |
| `noise`      | mean\|SHAP\| ≤ min(RANDOM_1, RANDOM_2) — no detectable signal |

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

from megumi.bansho import score_features

/Users/eligoze/miniforge3/envs/megumi-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Example 1 — Binary Classification

We generate a dataset with **20 features** split into three groups:

| Group | Count | Description |
|-------|-------|-------------|
| `informative_*` | 5 | Directly predictive of the target |
| `redundant_*`   | 3 | Linear combinations of the informative features — correlated with the target but carry no independent signal |
| `noise_*`       | 12 | Pure noise — statistically independent of the target |

We expect bansho to classify `informative_*` features as **predictive**, `redundant_*` as **marginal** or **noise**, and `noise_*` as **noise**.

In [2]:
N_SAMPLES    = 2_000
N_INFORMATIVE_CLF = 5
N_REDUNDANT  = 3
N_NOISE_CLF  = 12   # total = informative + redundant + noise = 20
N_FEATURES_CLF = N_INFORMATIVE_CLF + N_REDUNDANT + N_NOISE_CLF

X_clf, y_clf = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES_CLF,
    n_informative=N_INFORMATIVE_CLF,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    shuffle=False,
    random_state=42,
)

# make_classification places features in order: informative, redundant, noise
feature_names_clf = (
    [f"informative_{i+1}" for i in range(N_INFORMATIVE_CLF)]
    + [f"redundant_{i+1}"   for i in range(N_REDUNDANT)]
    + [f"noise_{i+1}"       for i in range(N_NOISE_CLF)]
)

df_clf = pd.DataFrame(X_clf, columns=feature_names_clf)
df_clf["target"] = y_clf

print(f"Shape: {df_clf.shape}")
print(f"Target distribution:\n{df_clf['target'].value_counts().to_string()}")
df_clf.head(3)

Shape: (2000, 21)
Target distribution:
target
0    1000
1    1000


,informative_1,informative_2,informative_3,informative_4,informative_5,redundant_1,redundant_2,redundant_3,noise_1,noise_2,...,noise_4,noise_5,noise_6,noise_7,noise_8,noise_9,noise_10,noise_11,noise_12,target
0,0.642928,-0.621520,2.663515,-0.599783,0.335670,0.886828,0.140331,-1.679709,-1.031106,-0.513127,...,0.909963,0.640027,0.356413,-0.661137,-0.200571,-0.077923,0.549574,1.508919,1.118338,0
1,-0.070846,2.901385,-0.832169,-1.501497,-0.212647,-0.616619,-3.085321,2.030037,-1.419384,0.223995,...,0.618860,-0.629234,2.914109,-1.277085,-1.083188,0.233595,2.116803,-0.836195,-0.265828,0
2,0.876182,2.158993,1.566156,-1.121890,-0.151822,-0.080111,-1.739280,0.330297,1.077294,-0.017862,...,0.565633,0.272518,1.030439,1.291469,0.307003,0.291733,0.214111,0.489131,0.055171,0


In [3]:
df_clf_train, df_clf_test = train_test_split(df_clf, test_size=0.2, random_state=42)

result_clf = score_features(
    df_clf_train,
    features=feature_names_clf,
    target="target",
    df_val=df_clf_test,
    random_state=42,
)
result_clf

,feature,predictive_power
0,informative_4,predictive
1,informative_2,predictive
2,informative_1,predictive
3,redundant_2,predictive
4,informative_5,predictive
5,redundant_1,predictive
6,informative_3,predictive
7,redundant_3,predictive
8,noise_8,predictive
9,noise_3,predictive


---

## Example 2 — Regression

We generate a dataset with **15 features** split into two groups:

| Group | Count | Description |
|-------|-------|-------------|
| `informative_*` | 5 | Linearly predictive of the target |
| `noise_*`       | 10 | Independent of the target |

A moderate noise level is added to make the task realistic. We expect bansho to classify `informative_*` as **predictive** and `noise_*` as **noise**.

In [ ]:
N_INFORMATIVE_REG = 5
N_NOISE_REG       = 10
N_FEATURES_REG    = N_INFORMATIVE_REG + N_NOISE_REG

X_reg, y_reg = make_regression(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES_REG,
    n_informative=N_INFORMATIVE_REG,
    noise=30.0,
    shuffle=False,
    random_state=42,
)

# make_regression places informative features first, then noise
feature_names_reg = (
    [f"informative_{i+1}" for i in range(N_INFORMATIVE_REG)]
    + [f"noise_{i+1}"     for i in range(N_NOISE_REG)]
)

df_reg = pd.DataFrame(X_reg, columns=feature_names_reg)
df_reg["target"] = y_reg

print(f"Shape: {df_reg.shape}")
print(f"Target stats:\n{df_reg['target'].describe().to_string()}")
df_reg.head(3)

In [ ]:
df_reg_train, df_reg_test = train_test_split(df_reg, test_size=0.2, random_state=42)

result_reg = score_features(
    df_reg_train,
    features=feature_names_reg,
    target="target",
    df_val=df_reg_test,
    random_state=42,
)
result_reg

---

## Summary

A side-by-side view of both runs: how many features in each true group ended up in each bansho tier.

In [ ]:
def build_summary(result: pd.DataFrame, dataset: str, task: str) -> pd.DataFrame:
    counts = (
        result["predictive_power"]
        .value_counts()
        .reindex(["predictive", "marginal", "noise"], fill_value=0)
        .to_frame(name="count")
        .T
    )
    counts.insert(0, "task", task)
    counts.insert(0, "dataset", dataset)
    counts.index = [""]
    return counts

summary = pd.concat([
    build_summary(result_clf, "classification", "binary"),
    build_summary(result_reg, "regression",     "continuous"),
], ignore_index=True)

summary